In [ ]:
# Cell 1: Load GitHub PAT from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
pat = secrets.get_secret('GITHUB_PAT')
os.environ['GITHUB_PAT'] = pat
print('PAT loaded OK')

In [ ]:
# Cell 2: Clone repo (branch dengan fixes)
import os
pat = os.environ['GITHUB_PAT']

# cd to safe dir first to avoid getcwd error when rm -rf deletes cwd
%cd /kaggle/working
!rm -rf EMA-SKD
!git clone https://{pat}@github.com/almaas-izdihar/ema-skd EMA-SKD
%cd EMA-SKD
!git checkout experiment/ablation-alignment
!git log --oneline -5

In [ ]:
# Cell 3: Verify GPU
!nvidia-smi

In [ ]:
# Cell 4: Run 1 — Baseline (L_CE only, no EHSKD)
!python main.py \
  --data_type cifar100 \
  --data_path /kaggle/working/data \
  --classifier_type ResNet18 \
  --batch_size 128 \
  --end_epoch 3 \
  --workers 4 \
  --seed 2024 \
  --experiment_type s0_baseline

In [ ]:
# Cell 5: Run 2 — EMA-SKD S0 (V7 code, pre-alignment)
!python main.py \
  --data_type cifar100 \
  --data_path /kaggle/working/data \
  --classifier_type ResNet18 \
  --batch_size 128 \
  --end_epoch 3 \
  --workers 4 \
  --seed 2024 \
  --beta 0.5 \
  --EHSKD \
  --experiment_type s0_emaskd_v7

In [ ]:
# Cell 8: Resource usage — GPU final state + training time from logs
import glob, re, os

# GPU current state
import subprocess
gpu_info = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu',
     '--format=csv,noheader,nounits'],
    capture_output=True, text=True
).stdout.strip()
print('=== GPU State (post-training) ===')
for i, line in enumerate(gpu_info.splitlines()):
    parts = [p.strip() for p in line.split(',')]
    if len(parts) >= 5:
        print(f'GPU {i}: {parts[0]}  VRAM: {parts[1]}/{parts[2]} MiB  '
              f'Util: {parts[3]}%  Temp: {parts[4]}°C')
print()

# Training time from log file timestamps (first and last line)
def get_training_duration(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    ts_pat = re.compile(r'\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})')
    times = [ts_pat.search(l) for l in lines]
    times = [t.group(1) for t in times if t]
    if len(times) < 2:
        return 'N/A'
    from datetime import datetime
    fmt = '%Y-%m-%d %H:%M:%S'
    dt = datetime.strptime(times[-1], fmt) - datetime.strptime(times[0], fmt)
    return str(dt)

print('=== Training Duration (from logs) ===')
for label, path in [('Baseline', baseline_log), ('EMA-SKD', ema_log)]:
    dur = get_training_duration(path)
    size = os.path.getsize(path) / 1024
    print(f'{label:10s}: {dur}  (log size: {size:.0f} KB)')

In [ ]:
# Cell 7: Training curves — Top-1, Val Loss, ECE per epoch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('EMA-SKD vs Baseline — CIFAR-100 / ResNet18', fontsize=13)

for ax, (col, title, lower_better) in zip(axes, [
    ('top1',     'Top-1 Accuracy (%)', False),
    ('val_loss', 'Val Loss',           True),
    ('ece',      'ECE (↓)',            True),
]):
    ax.plot(df_base.index, df_base[col], label='Baseline', marker='o', linewidth=1.5)
    ax.plot(df_ema.index,  df_ema[col],  label='EMA-SKD',  marker='s', linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eval_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: eval_curves.png')

# Crossover epoch: when does EMA-SKD first exceed baseline?
merged = df_ema[['top1']].rename(columns={'top1':'ema'}).join(
    df_base[['top1']].rename(columns={'top1':'base'}), how='inner')
crossover = merged[merged['ema'] > merged['base']]
if crossover.empty:
    print('EMA-SKD did not exceed baseline in any epoch — need more epochs or check config')
else:
    ep = crossover.index[0]
    print(f'EMA-SKD first exceeds baseline at epoch {ep}  '
          f'(EMA {crossover.loc[ep,"ema"]:.3f}% vs Base {crossover.loc[ep,"base"]:.3f}%)')

In [ ]:
# Cell 6: Metrics — parse logs and compare baseline vs EMA-SKD
import glob, re
import pandas as pd

def parse_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[val]' not in line:
                continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1)) if m else None
            ep = re.search(r'\[Epoch (\d+)\]', line)
            if not ep:
                continue
            rows.append({
                'epoch':    int(ep.group(1)),
                'top1':     g('val_top1_acc'),
                'top5':     g('val_top5_acc'),
                'val_loss': g('val_loss'),
                'ece':      g('ECE'),
                'aurc':     g('AURC'),
                'eaurc':    g('EAURC'),
            })
    return pd.DataFrame(rows).set_index('epoch')

def find_latest_log(keyword):
    matches = sorted(glob.glob(f'models/*{keyword}*/log/log.txt'))
    if not matches:
        raise FileNotFoundError(f'No log matching: {keyword}')
    return matches[-1]

# Auto-detect: baseline = no EHSKD, ema = EHSKD_True
all_logs = sorted(glob.glob('models/*/log/log.txt'))
base_logs = [p for p in all_logs if 'EHSKD_False' in p]
ema_logs  = [p for p in all_logs if 'EHSKD_True'  in p]

if not base_logs or not ema_logs:
    raise FileNotFoundError(f'Logs not found.\nBaseline: {base_logs}\nEMA: {ema_logs}')

baseline_log = base_logs[-1]
ema_log      = ema_logs[-1]

df_base = parse_log(baseline_log)
df_ema  = parse_log(ema_log)

print('Baseline log:', baseline_log)
print('EMA-SKD log :', ema_log)
print(f'Epochs — Baseline: {len(df_base)}  EMA-SKD: {len(df_ema)}')
print()

last_base = df_base.iloc[-1]
last_ema  = df_ema.iloc[-1]

summary = pd.DataFrame({
    'Metric':   ['Top-1 Acc (%)', 'Top-5 Acc (%)', 'ECE (↓)', 'AURC (↓)', 'EAURC (↓)'],
    'Baseline': [last_base.top1, last_base.top5, last_base.ece, last_base.aurc, last_base.eaurc],
    'EMA-SKD':  [last_ema.top1,  last_ema.top5,  last_ema.ece,  last_ema.aurc,  last_ema.eaurc],
})
summary['Δ'] = summary['EMA-SKD'] - summary['Baseline']
print(summary.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print()
print('Paper targets — Baseline: 75.55 ± 0.09  |  EMA-SKD: 79.19 ± 0.15')